# Hypothesis And Inference

## Hypothesis Testing
Hypothesis testing in ststistics is a formal process used to determine whether the sample data provides enough evidence to specific research.

- **Null Hypothesis($H_0$)** : The dafault position i.e nothing special is happening. Example Coin is fair (P = 0.50)
- **Alternative Hypothesis($H_1$)** : What we want to check against or the effect we want to prove. Example Coin is not fair (P $\neq$ 0.5) 

In [2]:
# Import libraries

import math
import statistics as stats
import statsmodels as statm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# Create function of Normal Probability Distributive Function 

def normal_pdf(x, mu=0, sigma=1):
    return math.exp(-(x - mu) ** 2 / 2 / sigma ** 2) / (math.sqrt(2 * math.pi) * sigma)

# Create function of Normal Cumulative Density Function

def normal_cdf(x, mu=0, sigma=1):
    return (1 + math.erf((x - mu) / math.sqrt(2) / sigma)) / 2

# Create function of Inverse of Normal Probability Distributive Function
# Find x where normal_cdf(x) approximate to p, using binary search

def inverse_normal_cdf(p, mu=0, sigma=1, tolerance=0.00001):
    if mu != 0 or sigma != 1:
        return mu + sigma * inverse_normal_cdf(p, tolerance=tolerance)
    low_z, hi_z = -10.0, 10.0
    while hi_z - low_z > tolerance:
        mid_z = (low_z + hi_z) / 2
        if normal_cdf(mid_z) < p:
            low_z = mid_z
        else:
            hi_z = mid_z
    return mid_z

### Flipping a Coin Example:

Before a local cricket tournament, the organisers want to check if the toss coin is fair.
- **$H_0$:** the coin is fair $\to$ probability of Heads p = 0.5
- **$H_1$:** the coin is not fair $\to$ p $\neq$ 0.5

We toss it n = 100 times and count the Heads (X). X is a **Binomial(n, p)** variable, which we can approximate with a **normal distribution**:
- mean = **$\mu$** = np
- standard deviation = **$\sigma$** = $\sqrt{np(1-p)}$

In [5]:
# Returns mu and sigma for a Binomial(n, p)

def normal_approximation_to_binomial(n, p):
    mu = n * p
    sigma = math.sqrt(n * p * (1 - p))
    return mu, sigma

mu_0, sigma_0 = normal_approximation_to_binomial(100, 0.5)
print("If the coin is fair: mean =", mu_0, ", standard deviation =", sigma_0)

If the coin is fair: mean = 50.0 , standard deviation = 5.0


In [6]:
# the normal cdf is the probability of being below a value

normal_probability_below = normal_cdf

# Above lo = not below lo
def normal_probability_above(lo, mu=0, sigma=1):
    return 1 - normal_cdf(lo, mu, sigma)

# Between = below hi, but not below lo
def normal_probability_between(lo, hi, mu=0, sigma=1):
    return normal_cdf(hi, mu, sigma) - normal_cdf(lo, mu, sigma)

# Outside = not between
def normal_probability_outside(lo, hi, mu=0, sigma=1):
    return 1 - normal_probability_between(lo, hi, mu, sigma)

print("P(X < 45)= ", round(normal_probability_below(45, mu_0, sigma_0), 3))
print("P(X > 55)= ", round(normal_probability_above(55, mu_0, sigma_0), 3))
print("P(45 < X < 55)= ", round(normal_probability_between(45, 55, mu_0, sigma_0), 3))
print("P(outside 45-55)= ", round(normal_probability_outside(45, 55, mu_0, sigma_0), 3))

P(X < 45)=  0.159
P(X > 55)=  0.159
P(45 < X < 55)=  0.683
P(outside 45-55)=  0.317


In [8]:
# Going backwards, finding the bounds
# Between which two values will X fall 95% of the time (if the coin is fair)

# Returns z such that P(Z <= z) = probability
def normal_upper_bound(probability, mu=0, sigma=1):
    return inverse_normal_cdf(probability, mu, sigma)

# Returns z such that P(Z >= z) = probability
def normal_lower_bound(probability, mu=0, sigma=1):
    return inverse_normal_cdf(1 - probability, mu, sigma)

# Returns the bounds (centred on the mean) that contain the given probability
def normal_two_sided_bounds(probability, mu=0, sigma=1):
    tail_probability = (1 - probability) / 2          # split the rest equally into two tails
    upper_bound = normal_lower_bound(tail_probability, mu, sigma)
    lower_bound = normal_upper_bound(tail_probability, mu, sigma)
    return lower_bound, upper_bound

lo, hi = normal_two_sided_bounds(0.95, mu_0, sigma_0)
print(f"95% of the time, a fair coin gives between {lo:.1f} and {hi:.1f} Heads in 100 tosses")

95% of the time, a fair coin gives between 40.2 and 59.8 Heads in 100 tosses
